In [1]:
import joblib 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# pick which model to explain — comment out the others
#TAG = "augmented"
#TAG = "freegen_qwen"
#TAG = "freegen_llama"
#TAG = "rewrite_qwen"
TAG = "rewrite_llama"

b = joblib.load(f"xgb_bundle_{TAG}.joblib")
clf, tfidf         = b["clf"], b["tfidf"]
feat_names, hand_names = b["feat_names"], b["hand_names"]
Xte, te_x, te_y    = b["Xte"], b["te_x"], np.array(b["te_y"])
hte, htr, tr_y     = b["hte"], b["htr"], np.array(b["tr_y"])
prob, pred         = np.array(b["prob"]), np.array(b["pred"])
print(f"Loaded {TAG}: {Xte.shape[0]} test notes, {len(feat_names)} features")

Loaded rewrite_llama: 400 test notes, 513 features


# Q4: Which feature is the decision most sensitive to? (sensitivity / pseudo-counterfactual)

In [2]:
from xgboost import XGBClassifier

clf_hand = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.5, reg_alpha=1.0, reg_lambda=2.0,
    eval_metric="logloss", random_state=42).fit(htr.values, tr_y)

# take one confidently-caught synthetic note
syn_rows = np.where((te_y == 1) & (pred == 1))[0]
i = int(syn_rows[np.argmax(prob[syn_rows])])
x0 = hte.values[i].copy()
print(f"{TAG}: original P(synthetic) = {clf_hand.predict_proba([x0])[0,1]:.3f}\n")

# set each feature to the real-class mean, one at a time; see which most reduces synthetic-ness
real_mean = htr.values[tr_y == 0].mean(axis=0)
rows = []
for j in range(len(hand_names)):
    x = x0.copy(); x[j] = real_mean[j]
    rows.append((hand_names[j], x0[j], real_mean[j], clf_hand.predict_proba([x])[0,1]))

print("Set feature → real-class mean (one at a time):")
for name, orig, tgt, p in sorted(rows, key=lambda t: t[3]):
    flag = "  → flips to REAL" if p < 0.5 else ""
    print(f"  {name:24s} {orig:9.2f} → {tgt:9.2f}   P(syn)={p:.3f}{flag}")

rewrite_llama: original P(synthetic) = 0.999

Set feature → real-class mean (one at a time):
  flesch                        8.37 →     42.43   P(syn)=0.917
  uppercase_word_ratio          0.03 →      0.15   P(syn)=0.922
  word_count                  583.00 →   1593.79   P(syn)=0.936
  sentence_count               46.00 →    148.58   P(syn)=0.992
  vocab_richness                0.72 →      0.49   P(syn)=0.995
  connector_density             0.51 →      0.09   P(syn)=0.998
  comma_density                 8.78 →      6.82   P(syn)=0.999
  paren_density                 0.92 →      2.13   P(syn)=0.999
  avg_sentence_length          12.89 →     11.42   P(syn)=0.999
  sentence_length_std          12.40 →     12.39   P(syn)=0.999
  sentence_length_cv            0.96 →      1.08   P(syn)=0.999
  colon_density                 8.32 →      8.39   P(syn)=0.999
  semicolon_density             0.00 →      0.19   P(syn)=0.999
